In [44]:
import pymcel as pc
import rebound as rb
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [85]:
#Creamos la simulación
sim = rb.Simulation()

fecha_inicio = '2029-01-01'

sim.add('Sun', date=fecha_inicio)

# El código 399 forzará a Horizons a devolver las coordenadas puras del Geocentro (Tierra) 
# El código 301 forzará a Horizons a devolver las coordenadas puras de la Luna (evitando el baricentro)
sim.add('399', date=fecha_inicio, hash='Earth')  # Tierra
sim.add('301', date=fecha_inicio, hash='Moon')   # Luna

sim.add('JWST', date=fecha_inicio)  
sim.add('Jupiter', date=fecha_inicio)  
sim.add('Saturn', date=fecha_inicio)  
sim.add('Apophis', date=fecha_inicio)

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for '399'... 
Found: Earth (399) 
Searching NASA Horizons for '301'... 
Found: Moon (301) 
Searching NASA Horizons for 'JWST'... 
Found: James Webb Space Telescope (spacecraft) (-170) 
Searching NASA Horizons for 'Jupiter'... 


c:\Users\ASUS\Downloads\mecanica_cel\mecanicacelev\Lib\site-packages\rebound\horizons.py:184: RuntimeWarning: Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.
  warnings.warn("Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.", RuntimeWarning)


Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Apophis'... 
Found: 99942 Apophis (2004 MN4) 


In [90]:
N_pasos = 3000 # Número de pasos de integración
t_max = 5 # 100 unidades de tiempo
ts = np.linspace(0, t_max, N_pasos) 

#Guardamos las posiciones y velocidades para los cuerpos
x_inis = [[] for _ in range(7)]
y_inis = [[] for _ in range(7)]
z_inis = [[] for _ in range(7)]
vx_inis = [[] for _ in range(7)]
vy_inis = [[] for _ in range(7)]
vz_inis = [[] for _ in range(7)]

#Integramos la simulación
for t in ts:
    sim.integrate(t)
    
    pos_tierra = sim.particles[1].xyz
    vel_tierra = sim.particles[1].vxyz

    for i, p in enumerate(sim.particles):
        # Guardamos posiciones relativas a la Tierra
        x_inis[i].append(p.x - pos_tierra[0])
        y_inis[i].append(p.y - pos_tierra[1])
        z_inis[i].append(p.z - pos_tierra[2])

        # De igual forma para las velocidades
        vx_inis[i].append(p.vx - vel_tierra[0])
        vy_inis[i].append(p.vy - vel_tierra[1])
        vz_inis[i].append(p.vz - vel_tierra[2])

print("Integración completada sin errores. Posiciones relativas a la Tierra en el inicio:")
print(x_inis[0][:2])

Integración completada sin errores. Posiciones relativas a la Tierra en el inicio:
[0.17830401285111022, 0.17996999147493725]


In [89]:
import plotly.graph_objects as go
import numpy as np

# Nombres en el mismo orden exacto en el que los agregaste arriba a `sim`
nombres = ['Sol', 'Tierra', 'Luna', 'JWST', 'Júpiter', 'Saturno', 'Apophis']
colores_planetas = ['#FFD700', 'blue', 'gray', 'purple', '#FF4500', '#DEB887', '#FF0000']

# Para esta visualización centrada en la tierra usaremos un zoom manual
# En lugar de recalcularlo desde Apophis que podría estar super lejos (distorsionando la velocidad aparente)
limite = 0.05  # Limite fijo para enfocar bien Cislunar + JWST

paso_frame = max(1, N_pasos // 100) # Reducimos cantidad de frames para mejorar rendimiento

# ------------------- Configuración para los gráficos ------------------------------

layout_base = dict(
    width=860, height=760,
    margin=dict(l=0, r=0, b=0, t=50),
    scene=dict(
        xaxis=dict(title="X [AU]", range=[-limite, limite]),
        yaxis=dict(title="Y [AU]", range=[-limite, limite]),
        zaxis=dict(title="Z [AU]", range=[-limite, limite]),
        # Es vital fijar aspectmode a 'cube' para que las órbitas no se distorsionen ni se aplasten
        aspectmode='cube',
        camera=dict(
            eye=dict(x=0.5, y=0.5, z=1) # El zoom lo ampliamos para que la caja 3D sea visible
        )
    ),
    updatemenus=[dict(
        type='buttons',
        direction="left",
        pad={"r": 10, "t": 87},
        showactive=True,
        x=0.1,
        xanchor="right",
        y=0,
        yanchor="top",
        buttons=[
            dict(label='▶ Play',
                method='animate',
                args=[None, dict(frame=dict(duration=40, redraw=True), 
                                fromcurrent=True, mode='immediate', transition=dict(duration=0))]), 
            dict(label='⏸ Pausa',
                method='animate',
                args=[[None], dict(frame=dict(duration=0, redraw=True),
                                    mode='immediate', transition=dict(duration=0))])
        ]
    )],
    sliders=[dict(
        active=0,
        yanchor="top",
        xanchor="left",
        currentvalue=dict(
            font=dict(size=14),
            prefix="Frame: ",
            visible=True,
            xanchor="right"
        ),
        transition=dict(duration=0), 
        pad=dict(b=10, t=50),
        len=0.9,
        x=0.1,
        y=0,
        steps=[dict(method='animate',
                    args=[[str(fi)], dict(mode='immediate',
                                        frame=dict(duration=0, redraw=True), transition=dict(duration=0))], 
                    label=str(fi))
            for fi in range(0, N_pasos, paso_frame)] 
    )]
)

# Trazo inicial: primero las órbitas y luego la animación de los cuerpos
trazos_base = []

# Primero agregamos las órbitas completas estáticas para los 7 cuerpos
for i in range(7):
    # Ocultamos la trayectoria del Sol, Júpiter y Saturno por defecto para no ensutiar el acercamiento
    # Pueden encenderse dando clic en la leyenda, pero aquí enfocamos en Apophis
    visible_val = True if i in [1, 2, 3, 6] else 'legendonly'
    
    trazos_base.append(
        go.Scatter3d(
            x=x_inis[i], 
            y=y_inis[i], 
            z=z_inis[i],
            mode='lines',
            line=dict(color=colores_planetas[i], width=2),
            name=f'Órbita {nombres[i]}',
            visible=visible_val
        )
    )

# Luego agregamos la burbuja inicial estática (animable)
# IMPORTANTE: Plotly indexa cada trazo. Hemos agregado 7 trazos de líneas (órbitas).
# Por tanto, este trazo de los marcadores ocupará SÓLO el índice 7.
trazos_base.append(
    go.Scatter3d(
        x=[x_inis[i][0] for i in range(7)],
        y=[y_inis[i][0] for i in range(7)],
        z=[z_inis[i][0] for i in range(7)],
        mode='markers+text',
        text=nombres, textposition="top center",
        marker=dict(size=4, color=colores_planetas),
        showlegend=False
    )
)

# Animación del sistema solar SÓLO inyectando cambios al trazo de índice 7
fig1 = go.Figure(
    data=trazos_base,
    frames=[go.Frame(
        name=str(k),
        data=[
            go.Scatter3d(
                x=[x_inis[i][k] for i in range(7)],
                y=[y_inis[i][k] for i in range(7)],
                z=[z_inis[i][k] for i in range(7)]
            )
        ],
        traces=[7]
    ) for k in range(0, N_pasos, paso_frame)]
)

fig1.update_layout(**layout_base)
fig1.update_layout(title="Acercamiento: Tierra, Luna, JWST y Apophis")
fig1.show()